# 17 · Item retrieval index for the prototype (MiniLM, price and category filters)
Owner: Disha. Builds the vector index the prototype's *Preference and Retrieval* agent searches: exact cosine
search over the **MiniLM item-text embeddings** (Abhinita's verified full-catalog vectors, 25,903 items), with
**price, category and brand filters**, an exclude list (e.g. items the user already has), and an empty result for
"no suitable match". Runs on a CPU (Colab CPU runtime is enough, about 5 minutes); the prototype's EC2 server needs
no GPU either.

### Training-split data only
The index holds catalog metadata only: the `item_text` field (Title, Brand, Category, Features, Description, Price
band). No review text and no interaction data, so nothing from validation or test can reach a search result.

### Inputs (checked against the frozen manifest and Abhinita's verification)
* `298A/04_Data/NewDataRunFiles/processed/item_text.parquet`
* `298A/model1_item_text/minilm_full_pinned_*/item_text_emb_minilm.npy` (+ `metadata.json`, `verification.json`)
* Optional, for the profile demo: `sequences.parquet` and the pilot's `profiles_minilm.npz`

### Output: `298A/retrieval_index/item_index_minilm_v1/` on the shared Drive
| File | Contents |
|---|---|
| `embeddings.npy` | 25,903 × 384 float32, L2-normalized, one row per item in `items.parquet` order |
| `items.parquet` | `item_idx`, `title`, `brand`, `category_path`, `cat_l2`, `cat_l3`, `price_num`, `price_band` |
| `index_meta.json` | encoder and commit, source files and hashes, filters, checksums |
| `build_report.json` | all checks and their results |

### Loading it (Ruchika / the prototype)
```python
from src.retrieval.item_index import ItemIndex
index = ItemIndex.load('<298A>/retrieval_index/item_index_minilm_v1')
index.search('gentle moisturizer for dry skin', k=10, price_max=25, categories=['Face'])
index.search_vector(profile_vector, k=10, exclude_items=train_items)    # a MiniLM user profile works directly
index.categories(level=2)                                               # valid category names for the agent
```

In [1]:
# ============================== SETTINGS ==============================
PROJECT_ROOT = None        # the shared '298A' folder; None = find it (Colab: /content/drive/MyDrive/298A)
EMBEDDING_DIR = None       # Abhinita's MiniLM output folder; None = the newest minilm_full_pinned_* with a PASS
INDEX_NAME = 'item_index_minilm_v1'
ENCODER_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
ENCODER_REVISION = '1110a243fdf4706b3f48f1d95db1a4f5529b4d41'       # same commit as the item vectors
EXPECTED_EMBEDDING_SHA256 = '630aa65eb5edbb0482d96bc14c5e8d42af606fd284de214119c92e6c6c81082a'
PILOT_PROFILES = 'model1_profiles/pilot_v2_qwen35_9b/profiles_minilm.npz'   # optional demo
SEED = 42
# ======================================================================

In [2]:
import sys, subprocess, importlib.metadata as md
def _ver(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None
PINNED = {'transformers': '5.24.0', 'sentence-transformers': '6.3.0'}
need = [f'{n}=={v}' for n, v in PINNED.items() if _ver(n) != v]
if need:
    print('Installing', need, '...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *need], check=True)
    if any(m in sys.modules for m in ('transformers', 'sentence_transformers', 'huggingface_hub', 'tokenizers')):
        raise RuntimeError('Installed new library versions. RESTART the runtime/kernel now, then Run all again.')
print({n: _ver(n) for n in ('torch', 'transformers', 'sentence-transformers', 'numpy', 'pandas', 'pyarrow')})

Installing ['transformers==5.24.0', 'sentence-transformers==6.3.0'] ...
{'torch': '2.14.1+cu130', 'transformers': '5.24.0', 'sentence-transformers': '6.3.0', 'numpy': '2.4.6', 'pandas': '2.2.3', 'pyarrow': '23.0.1'}


## 1. Locations

In [3]:
import os, json, re, time, glob, hashlib, datetime, shutil, tempfile
from pathlib import Path
import numpy as np
import pandas as pd

for k in ('PROJECT_ROOT', 'EMBEDDING_DIR', 'ENCODER_MODEL', 'ENCODER_REVISION'):     # local testing only
    if os.environ.get(f'INDEX_{k}'):
        globals()[k] = os.environ[f'INDEX_{k}']
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

DATA_REL = Path('04_Data') / 'NewDataRunFiles' / 'processed'
def find_project_root():
    home = Path.home()
    bases = [Path('/content/drive/MyDrive'), *Path('/content/drive/Shareddrives').glob('*')]
    bases += [Path(f'{d}:/') / sub for d in 'DEFGHIJKLMNOPQRSTUVWXYZ' for sub in ('My Drive', 'Shared drives/*')]
    bases += [*home.glob('Library/CloudStorage/GoogleDrive-*/My Drive'), *home.glob('Library/CloudStorage/GoogleDrive-*/Shared drives/*'),
              home / 'Google Drive' / 'My Drive', home / 'GoogleDrive', home / 'gdrive', home / 'Google Drive', home]
    found = []
    for b in bases:
        for base in (glob.glob(str(b)) if '*' in str(b) else [str(b)]):
            for cand in (Path(base) / '298A', *Path(base).glob('*/298A')):
                try:
                    if (cand / DATA_REL / 'item_text.parquet').exists():
                        found.append(cand.resolve())
                except OSError:
                    pass
    found = sorted(set(found))
    if len(found) > 1 and len({(r / DATA_REL / 'item_text.parquet').stat().st_size for r in found}) == 1:
        found = found[:1]
    return found

if PROJECT_ROOT is None:
    roots = find_project_root()
    if len(roots) != 1:
        raise FileNotFoundError('Could not find the shared 298A folder (found: ' + (', '.join(map(str, roots)) or 'none') +
                                '). Set PROJECT_ROOT in the settings cell.')
    PROJECT_ROOT = roots[0]
PROJECT_ROOT = Path(PROJECT_ROOT)
DATA_DIR = PROJECT_ROOT / DATA_REL

if EMBEDDING_DIR is None:
    cands = []
    for d in sorted((PROJECT_ROOT / 'model1_item_text').glob('minilm_full_pinned_*')):
        try:
            if d.is_dir() and json.loads((d / 'verification.json').read_text()).get('status') == 'PASS':
                cands.append(d)
        except (OSError, ValueError):
            pass
    if not cands:
        raise FileNotFoundError(f'No verified MiniLM folder (minilm_full_pinned_* with status PASS) in '
                                f'{PROJECT_ROOT / "model1_item_text"}. Set EMBEDDING_DIR in the settings cell.')
    EMBEDDING_DIR = cands[-1]
EMBEDDING_DIR = Path(EMBEDDING_DIR)
OUT_DIR = PROJECT_ROOT / 'retrieval_index' / INDEX_NAME
OUT_DIR.parent.mkdir(parents=True, exist_ok=True)
(OUT_DIR.parent / '.write_test').write_text('ok'); (OUT_DIR.parent / '.write_test').unlink()
print('298A folder     :', PROJECT_ROOT)
print('Item text       :', DATA_DIR / 'item_text.parquet')
print('Item embeddings :', EMBEDDING_DIR)
print('Index output    :', OUT_DIR, '(exists: rebuild is checked against it)' if OUT_DIR.exists() else '(new)')

Mounted at /content/drive
298A folder     : /content/drive/.shortcut-targets-by-id/1K2gvVTjU-77GWGMI14ixFt5F57IaijZB/298A
Item text       : /content/drive/.shortcut-targets-by-id/1K2gvVTjU-77GWGMI14ixFt5F57IaijZB/298A/04_Data/NewDataRunFiles/processed/item_text.parquet
Item embeddings : /content/drive/.shortcut-targets-by-id/1K2gvVTjU-77GWGMI14ixFt5F57IaijZB/298A/model1_item_text/minilm_full_pinned_20261112T002342156654Z
Index output    : /content/drive/.shortcut-targets-by-id/1K2gvVTjU-77GWGMI14ixFt5F57IaijZB/298A/retrieval_index/item_index_minilm_v1 (exists: rebuild is checked against it)


## 2. Check the inputs
`item_text.parquet` must be the frozen version. The embedding file must be the one Abhinita verified, built from
that frozen data, with the same encoder and commit used here for queries.

In [4]:
ITEM_TEXT_SHA256 = '39dbd1ac817df3383def7c677fca830aaf36cfe470e1c0612c543486e6b8909d'
SEQUENCES_SHA256 = '0a55f2c375c06594b322d1e23dcd49d975ac7f8e1f7db12e4cc8bea59128186a'
MANIFEST_SHA256 = 'e4b7801c0d9ea27d0b39a6af253ac154d6f4a7181d8a7c1dc31d651a050af0e5'
if os.environ.get('INDEX_EXPECTED_HASHES_JSON'):                          # local testing with synthetic data only
    _h = json.loads(Path(os.environ['INDEX_EXPECTED_HASHES_JSON']).read_text())
    ITEM_TEXT_SHA256, SEQUENCES_SHA256 = _h['item_text.parquet'], _h['sequences.parquet']
    MANIFEST_SHA256, EXPECTED_EMBEDDING_SHA256 = _h['manifest'], _h['embedding']

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()

assert sha256_file(DATA_DIR / 'item_text.parquet') == ITEM_TEXT_SHA256, 'item_text.parquet is not the frozen version'
emb_meta = json.loads((EMBEDDING_DIR / 'metadata.json').read_text())
emb_ver = json.loads((EMBEDDING_DIR / 'verification.json').read_text())
EMB_FILE = EMBEDDING_DIR / emb_meta['file']
EMB_SHA = sha256_file(EMB_FILE)
assert emb_meta['file'] == 'item_text_emb_minilm.npy', emb_meta['file']
assert EMB_SHA == emb_meta['embedding_sha256'] == emb_ver['embedding_sha256'] == EXPECTED_EMBEDDING_SHA256, \
    'embedding file does not match its metadata, its verification or the expected checksum'
assert emb_ver['status'] == 'PASS' and emb_meta.get('complete_catalog') is True
assert emb_meta['manifest_sha256'] == MANIFEST_SHA256, 'embeddings were built from a different frozen dataset'
flat = json.dumps(emb_meta)
assert ENCODER_MODEL in flat and ENCODER_REVISION in flat, 'embeddings were built with a different encoder or commit'
assert emb_meta.get('text_column', 'item_text') == 'item_text', 'embeddings must come from the metadata-only item_text'
print('PASS item_text.parquet is frozen; embeddings', EMB_SHA[:12], 'verified, same frozen data, same encoder',
      ENCODER_MODEL, '@', ENCODER_REVISION[:8])

PASS item_text.parquet is frozen; embeddings 630aa65eb5ed verified, same frozen data, same encoder sentence-transformers/all-MiniLM-L6-v2 @ 1110a243


## 3. Load the vectors and build the item table

In [5]:
matrix = np.load(EMB_FILE, allow_pickle=False)
cols = ['item_idx', 'title', 'brand', 'cat_path', 'cat_l1', 'cat_l2', 'cat_l3', 'price_num', 'item_text']
it = pd.read_parquet(DATA_DIR / 'item_text.parquet', columns=cols).sort_values('item_idx').reset_index(drop=True)
N = len(it)
assert it.item_idx.tolist() == list(range(1, N + 1)), 'item ids must be 1..n'
assert matrix.shape == (N + 1, 384) and matrix.dtype == np.float32, matrix.shape
assert not np.count_nonzero(matrix[0]), 'row 0 is the padding row and must be zero'
emb = np.ascontiguousarray(matrix[1:])                                    # row r = item_idx r + 1
norms = np.linalg.norm(emb, axis=1)
assert np.isfinite(emb).all() and np.allclose(norms, 1.0, atol=1e-4), 'vectors must be finite and unit length'

def price_band(p):                                                        # same bands as notebook 06
    if pd.isna(p): return ''
    for hi, label in [(10, 'under $10'), (25, '$10-25'), (50, '$25-50'), (100, '$50-100')]:
        if p < hi: return label
    return 'over $100'
def cat_display(path):                                                    # same as notebook 06: drop the root level
    if path is None or (not isinstance(path, str) and pd.isna(path)):
        return ''
    parts = [p for p in str(path).split(' > ') if p]
    return ' > '.join(parts[1:] if len(parts) > 1 else parts)

items = pd.DataFrame({
    'item_idx': it.item_idx.astype('int64'),
    'title': it.title.fillna('').astype(str),
    'brand': it.brand.fillna('').astype(str),
    'category_path': it.cat_path.map(cat_display),
    'cat_l2': it.cat_l2, 'cat_l3': it.cat_l3,
    'price_num': pd.to_numeric(it.price_num, errors='coerce').astype('float64'),
})
items['price_band'] = items.price_num.map(price_band)
roots = it.cat_path.dropna().str.split(' > ').str[0].value_counts()
STATS = {'items': int(N), 'with_price': int(items.price_num.notna().sum()),
         'with_price_pct': round(100 * items.price_num.notna().mean(), 1),
         'price_min': float(items.price_num.min()), 'price_median': float(items.price_num.median()),
         'price_max': float(items.price_num.max()),
         'with_category': int((items.category_path != '').sum()), 'cat_path_roots': roots.to_dict(),
         'cat_l1_values': it.cat_l1.value_counts().head(5).to_dict(),
         'top_cat_l2': items.cat_l2.value_counts().head(10).to_dict()}
print(json.dumps(STATS, indent=1))

{
 "items": 25903,
 "with_price": 16502,
 "with_price_pct": 63.7,
 "price_min": 0.67,
 "price_median": 19.9,
 "price_max": 449.77,
 "with_category": 25903,
 "cat_path_roots": {
  "Beauty & Personal Care": 25903
 },
 "cat_l1_values": {
  "Skin Care": 25903
 },
 "top_cat_l2": {
  "Face": 12817,
  "Body": 8214,
  "Sunscreens & Tanning Products": 2020,
  "Lip Care": 1294,
  "Eyes": 1199,
  "Sets & Kits": 247,
  "Maternity": 34
 }
}


## 4. Are queries encoded in the same space as the items?
A few items are re-encoded here with the same pinned MiniLM (through `sentence-transformers`, which is what the
index uses for text queries) and compared with Abhinita's stored vectors. They must be practically identical
(cosine ≥ 0.999), including the longest item text, which tests the same 256-token truncation.

In [6]:
import torch
from sentence_transformers import SentenceTransformer
torch.manual_seed(SEED)
local_encoder = Path(ENCODER_MODEL).exists()
encoder = SentenceTransformer(ENCODER_MODEL, revision=None if local_encoder else ENCODER_REVISION, device='cpu')
rng = np.random.default_rng(SEED)
check_rows = sorted({0, N // 2, N - 1, int(it.item_text.str.len().argmax()), *rng.choice(N, 4, replace=False).tolist()})
re_emb = encoder.encode(it.item_text.iloc[check_rows].tolist(), normalize_embeddings=True, convert_to_numpy=True)
cos = (re_emb * emb[check_rows]).sum(axis=1)
SAME_SPACE = {'items_checked': [int(it.item_idx[r]) for r in check_rows], 'min_cosine': round(float(cos.min()), 6),
              'max_seq_length': int(encoder.max_seq_length or 0)}
print(SAME_SPACE)
assert cos.min() >= 0.999, 'query encoder does not reproduce the stored item vectors: wrong model, commit or settings'
print('PASS queries and items are in the same embedding space')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

{'items_checked': [1, 2312, 4573, 11369, 12952, 16955, 20047, 25903], 'min_cosine': 1.0, 'max_seq_length': 256}
PASS queries and items are in the same embedding space


## 5. The index code (identical to `src/retrieval/item_index.py`)

In [7]:
"""(copied from src/retrieval/item_index.py) Item retrieval index for the prototype: exact cosine search over the MiniLM item-text embeddings,
with price, category and brand filters. CPU only; no extra index library is needed for 25,903 items.

The index holds only catalog metadata text (title, brand, category, features, description, price band),
never review or interaction text, so it carries no validation or test information.

Dependencies: numpy, pandas and pyarrow for loading and vector search; sentence-transformers only for
text queries (search / encode_query), loaded on the first text query.

Usage:
    from src.retrieval.item_index import ItemIndex
    index = ItemIndex.load('<298A>/retrieval_index/item_index_minilm_v1')
    hits = index.search('gentle moisturizer for dry skin', k=10, price_max=25, categories=['Face'])
    hits = index.search_vector(profile_vector, k=10, exclude_items=train_items)   # e.g. a MiniLM user profile
"""
import difflib
import hashlib
import json
import math
import numbers
from pathlib import Path

import numpy as np
import pandas as pd

INDEX_FORMAT = 'item_index_v1'
FILES = ('embeddings.npy', 'items.parquet', 'index_meta.json')
ITEM_COLUMNS = ['item_idx', 'title', 'brand', 'category_path', 'cat_l2', 'cat_l3', 'price_num', 'price_band']


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()


def category_levels(category_path):
    # 'Skin Care > Face > Creams & Moisturizers' -> ['skin care', 'face', 'creams & moisturizers']; missing -> []
    if category_path is None or (not isinstance(category_path, str) and pd.isna(category_path)):
        return []
    return [p.strip().lower() for p in str(category_path).split(' > ') if p.strip()]


class ItemIndex:
    def __init__(self, embeddings, items, meta, encoder=None):
        embeddings = np.asarray(embeddings, dtype=np.float32)
        if embeddings.ndim != 2 or len(embeddings) != len(items):
            raise ValueError('embeddings must be a 2-D matrix with one row per item')
        if not np.isfinite(embeddings).all() or not np.allclose(np.linalg.norm(embeddings, axis=1), 1.0, atol=1e-3):
            raise ValueError('embeddings must be finite and L2-normalized')
        missing = [c for c in ITEM_COLUMNS if c not in items.columns]
        if missing:
            raise ValueError(f'items table is missing columns {missing}')
        if not items.item_idx.is_unique:
            raise ValueError('item_idx must be unique')
        self.embeddings = embeddings
        self.items = items.reset_index(drop=True)
        self.meta = meta
        self.item_ids = self.items.item_idx.to_numpy(dtype=np.int64)
        self.row_of = {int(i): r for r, i in enumerate(self.item_ids)}
        self.price = self.items.price_num.to_numpy(dtype=np.float64)
        self.brand_lower = self.items.brand.fillna('').astype(str).str.strip().str.lower().to_numpy()
        self.levels = [category_levels(p) for p in self.items.category_path]
        self._encoder = encoder

    # ------------------------------------------------------------------ loading
    @classmethod
    def load(cls, directory, verify=True, encoder=None):
        """Load an index folder. encoder: an already loaded SentenceTransformer to reuse for text queries
        (it must be the model and commit in index_meta.json); by default it is loaded on the first text query."""
        directory = Path(directory)
        meta = json.loads((directory / 'index_meta.json').read_text())
        if meta.get('format') != INDEX_FORMAT:
            raise ValueError(f'not an {INDEX_FORMAT} folder: {directory}')
        if verify:
            for name in ('embeddings.npy', 'items.parquet'):
                if sha256_file(directory / name) != meta['sha256'][name]:
                    raise ValueError(f'{name} does not match its checksum in index_meta.json')
        emb = np.load(directory / 'embeddings.npy', allow_pickle=False)
        items = pd.read_parquet(directory / 'items.parquet')
        if emb.shape != (meta['n_items'], meta['dim']) or len(items) != meta['n_items']:
            raise ValueError('index files do not match index_meta.json')
        return cls(emb, items, meta, encoder=encoder)

    # ------------------------------------------------------------------ helpers
    def categories(self, level=None):
        """Category names the filter accepts, with item counts (level 2 = e.g. Face, Body; level 3 below that)."""
        col = {2: 'cat_l2', 3: 'cat_l3'}.get(level)
        if col:
            return self.items[col].dropna().value_counts().rename_axis(col).reset_index(name='n_items')
        names = pd.Series([lvl for levels in self.levels for lvl in levels[1:]])
        return names.value_counts().rename_axis('category').reset_index(name='n_items')

    def _encoder_model(self):
        if self._encoder is None:
            from sentence_transformers import SentenceTransformer
            enc = self.meta['encoder']
            self._encoder = SentenceTransformer(enc['model'], revision=enc.get('revision'), device='cpu')
        return self._encoder

    def encode_query(self, text):
        if not isinstance(text, str) or not text.strip():
            raise ValueError('query text must be a non-empty string')
        prefix = self.meta['encoder'].get('query_prefix', '')
        v = self._encoder_model().encode([prefix + text.strip()], normalize_embeddings=True, convert_to_numpy=True)
        return v[0].astype(np.float32)

    def filter_mask(self, price_min=None, price_max=None, categories=None, brands=None, exclude_items=None,
                    include_unpriced=False):
        """Boolean mask over items. Price bounds are inclusive. Items without a price are dropped when a price bound
        is given, unless include_unpriced=True. A category matches any level of the item's category path, or a
        path prefix such as 'Face > Creams & Moisturizers' (case-insensitive). Unknown category or brand names
        raise ValueError with close matches, so a caller never filters on a misspelled name by accident."""
        mask = np.ones(len(self.items), dtype=bool)
        for label, bound in (('price_min', price_min), ('price_max', price_max)):
            if bound is not None and (isinstance(bound, bool) or not isinstance(bound, numbers.Real)
                                      or not math.isfinite(bound)):
                raise ValueError(f'{label} must be a finite number, got {bound!r}')
        if price_min is not None or price_max is not None:
            if price_min is not None and price_max is not None and price_min > price_max:
                raise ValueError('price_min is larger than price_max')
            has = ~np.isnan(self.price)
            ok = has.copy()
            if price_min is not None:
                ok &= np.where(has, self.price >= price_min, False)
            if price_max is not None:
                ok &= np.where(has, self.price <= price_max, False)
            mask &= ok | (~has if include_unpriced else False)
        if categories:
            wanted = [category_levels(c) for c in ([categories] if isinstance(categories, str) else categories)]
            known = {lvl for levels in self.levels for lvl in levels}
            for w in wanted:
                if not w:
                    raise ValueError('empty category name')
                for level in w:                                  # every level of a path must exist
                    if level not in known:
                        close = difflib.get_close_matches(level, sorted(known), n=5, cutoff=0.5)
                        raise ValueError(f'unknown category {level!r}; close matches: {close}')
                if len(w) > 1 and not any(levels[s:s + len(w)] == w for levels in self.levels
                                          for s in range(len(levels) - len(w) + 1)):
                    raise ValueError(f'category path {" > ".join(w)!r} does not occur in the catalog '
                                     '(each level exists, but not in this order)')
            def match(levels):
                for w in wanted:
                    if len(w) == 1 and w[0] in levels:
                        return True
                    if len(w) > 1 and any(levels[s:s + len(w)] == w for s in range(len(levels) - len(w) + 1)):
                        return True
                return False
            mask &= np.fromiter((match(levels) for levels in self.levels), dtype=bool, count=len(self.levels))
        if brands:
            wanted = {b.strip().lower() for b in ([brands] if isinstance(brands, str) else brands)}
            known = set(self.brand_lower)
            unknown = sorted(wanted - known)
            if unknown:
                close = {u: difflib.get_close_matches(u, sorted(known), n=3, cutoff=0.6) for u in unknown}
                raise ValueError(f'unknown brand(s); close matches: {close}')
            mask &= np.isin(self.brand_lower, list(wanted))
        if exclude_items is not None:
            rows = [self.row_of[int(i)] for i in exclude_items if int(i) in self.row_of]
            mask[rows] = False
        return mask

    # ------------------------------------------------------------------ search
    def search_vector(self, vector, k=10, min_score=None, **filters):
        """Top-k items by cosine similarity to a vector in the index's embedding space, after the filters.
        Returns a DataFrame (rank, item_idx, score and item metadata), possibly empty: an empty result means
        "no suitable match" for these filters or this min_score."""
        if not isinstance(k, (int, np.integer)) or k < 1:
            raise ValueError('k must be a positive integer')
        v = np.asarray(vector, dtype=np.float32).reshape(-1)
        if v.shape[0] != self.embeddings.shape[1] or not np.isfinite(v).all() or np.linalg.norm(v) == 0:
            raise ValueError(f'query vector must be finite, non-zero and of length {self.embeddings.shape[1]}')
        v = v / np.linalg.norm(v)
        rows = np.flatnonzero(self.filter_mask(**filters))
        if len(rows) == 0:
            return self._result(rows, np.array([], dtype=np.float32))
        with np.errstate(all='ignore'):                 # numpy with Apple Accelerate can warn spuriously here
            scores = self.embeddings[rows] @ v
        if not np.isfinite(scores).all():               # so check the result itself instead
            raise FloatingPointError('non-finite similarity scores')
        top = np.argsort(-scores, kind='stable')[:k]
        rows, scores = rows[top], scores[top]
        if min_score is not None:
            keep = scores >= min_score
            rows, scores = rows[keep], scores[keep]
        return self._result(rows, scores)

    def search(self, query, k=10, min_score=None, **filters):
        """Top-k items for a text query (encoded with the same pinned MiniLM as the items), after the filters."""
        return self.search_vector(self.encode_query(query), k=k, min_score=min_score, **filters)

    def _result(self, rows, scores):
        out = self.items.iloc[rows][ITEM_COLUMNS].copy().reset_index(drop=True)
        out.insert(0, 'rank', np.arange(1, len(out) + 1))
        out.insert(2, 'score', np.round(scores.astype(np.float64), 6))
        return out


def build_index(embeddings, items, meta, out_dir):
    """Write an index folder (embeddings.npy, items.parquet, index_meta.json) and return its checksums."""
    ItemIndex(embeddings, items, meta)                          # validates before anything is written
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / 'embeddings.npy', np.asarray(embeddings, dtype=np.float32), allow_pickle=False)
    items[ITEM_COLUMNS].reset_index(drop=True).to_parquet(out_dir / 'items.parquet', index=False)
    meta = dict(meta, format=INDEX_FORMAT, n_items=int(len(items)), dim=int(np.asarray(embeddings).shape[1]),
                sha256={n: sha256_file(out_dir / n) for n in ('embeddings.npy', 'items.parquet')})
    (out_dir / 'index_meta.json').write_text(json.dumps(meta, indent=2))
    return meta['sha256']

## 6. Build the index and save it to the shared Drive
The index is built in a temporary folder and checked first. If the output folder already exists, the new build
must give the same vectors and item table; an existing index is never overwritten with different content.

In [8]:
INDEX_META = {
    'description': 'Exact cosine search over MiniLM item-text embeddings (metadata-only item_text) with price, '
                   'category and brand filters, for the 298A prototype.',
    'created_utc': datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
    'encoder': {'model': str(ENCODER_MODEL), 'revision': None if local_encoder else ENCODER_REVISION, 'query_prefix': '', 'pooling': 'mean', 'normalized': True,
                'max_seq_length': SAME_SPACE['max_seq_length']},
    'source': {'item_text_sha256': ITEM_TEXT_SHA256, 'manifest_sha256': MANIFEST_SHA256,
               'embedding_dir': EMBEDDING_DIR.name, 'embedding_sha256': EMB_SHA, 'text_column': 'item_text',
               'text_fields': ['Title', 'Brand', 'Category', 'Features', 'Description', 'Price band'],
               'uses_reviews_or_interactions': False},
    'search': 'exact (brute-force) inner product on L2-normalized vectors; ties broken by item order',
    'filters': {'price_min / price_max': 'inclusive bounds on price_num (USD); items without a price are dropped '
                                         'when a bound is given unless include_unpriced=True',
                'categories': 'any level of category_path, or a path prefix such as "Face > Creams & Moisturizers"; '
                              'case-insensitive; unknown names raise ValueError with close matches',
                'brands': 'exact brand names, case-insensitive', 'exclude_items': 'item_idx values to leave out',
                'min_score': 'drop results below this cosine; an empty result means no suitable match'},
    'stats': STATS, 'versions': {n: _ver(n) for n in ('numpy', 'pandas', 'pyarrow', 'sentence-transformers', 'transformers')},
}
tmp_dir = Path(tempfile.mkdtemp()) / INDEX_NAME
CHECKSUMS = build_index(emb, items, INDEX_META, tmp_dir)
built = ItemIndex.load(tmp_dir)
assert np.array_equal(built.embeddings, emb) and built.items.item_idx.tolist() == items.item_idx.tolist()

if (OUT_DIR / 'index_meta.json').exists():                               # index_meta.json is written last
    existing = ItemIndex.load(OUT_DIR)
    same = np.array_equal(existing.embeddings, emb) and existing.items.equals(built.items)
    assert same, (f'{OUT_DIR} already holds a DIFFERENT index. Never overwrite a released index: '
                  'set INDEX_NAME to a new version (e.g. item_index_minilm_v2).')
    print('Existing index has the same content; keeping it.')
else:
    OUT_DIR.mkdir(parents=True, exist_ok=True)                           # new, or an earlier copy that never finished
    for name in ('embeddings.npy', 'items.parquet', 'index_meta.json'):  # index_meta.json last = complete
        shutil.copyfile(tmp_dir / name, OUT_DIR / name)
    print('Index written to', OUT_DIR)
index = ItemIndex.load(OUT_DIR, encoder=encoder)
assert int((index.items.category_path == '').sum()) == int(it.cat_path.isna().sum()), 'missing categories must stay empty'
assert 'nan' not in {l for ls in index.levels for l in ls}
print(f'Loaded: {index.meta["n_items"]:,} items x {index.meta["dim"]} dims; files:',
      sorted(p.name for p in OUT_DIR.iterdir()))

Existing index has the same content; keeping it.
Loaded: 25,903 items x 384 dims; files: ['build_report.json', 'embeddings.npy', 'index_meta.json', 'items.parquet']


## 7. Check the search results
* **Exact:** 50 random queries with random filters give the same items, in the same order, as a plain
  filter-then-sort over all items.
* **Filters are always respected** in every result.
* **Self-retrieval:** searching for an item's own title should find that item near the top
  (a sanity check that queries and items match; not the 298B retrieval evaluation).

In [9]:
def naive(v, k, price_min=None, price_max=None, categories=None, exclude_items=None, include_unpriced=False):
    df = items.copy()
    df['score'] = emb @ (v / np.linalg.norm(v))
    if price_min is not None or price_max is not None:
        ok = df.price_num.notna()
        if price_min is not None: ok &= df.price_num >= price_min
        if price_max is not None: ok &= df.price_num <= price_max
        df = df[ok | (df.price_num.isna() & include_unpriced)]
    if categories:
        wanted = [c.lower() for c in categories]
        df = df[df.category_path.map(lambda p: any(w in [x.strip().lower() for x in str(p).split(' > ')] for w in wanted))]
    if exclude_items:
        df = df[~df.item_idx.isin(exclude_items)]
    return df.sort_values('score', ascending=False, kind='stable').item_idx.head(k).tolist()

known_levels = {lvl for levels in index.levels for lvl in levels}
top_l2 = [c for c in index.categories(level=2).cat_l2.head(10) if str(c).strip().lower() in known_levels] \
         or index.categories().category.head(5).tolist()
n_exact, n_filter_ok = 0, 0
for t in range(50):
    v = emb[rng.integers(N)] + 0.3 * rng.normal(size=emb.shape[1]).astype(np.float32)
    f = {}
    if t % 2: f['price_max'] = float(rng.choice([10, 25, 50]))
    if t % 3 == 0: f['price_min'] = 5.0
    if t % 4 == 1 and top_l2: f['categories'] = [str(rng.choice(top_l2))]
    if t % 5 == 2: f['include_unpriced'] = True
    if t % 6 == 3: f['exclude_items'] = rng.choice(items.item_idx, 20, replace=False).tolist()
    got = index.search_vector(v, k=10, **f)
    n_exact += got.item_idx.tolist() == naive(v, 10, **f)
    pm = got.price_num
    ok = True
    if 'price_max' in f: ok &= bool(((pm <= f['price_max']) | (pm.isna() & f.get('include_unpriced', False))).all())
    if 'price_min' in f: ok &= bool(((pm >= f['price_min']) | (pm.isna() & f.get('include_unpriced', False))).all())
    if 'categories' in f: ok &= bool(got.category_path.str.lower().str.contains(f['categories'][0].lower(), regex=False).all())
    if 'exclude_items' in f: ok &= not set(got.item_idx) & set(f['exclude_items'])
    n_filter_ok += ok
EXACT = {'queries': 50, 'same_as_brute_force': int(n_exact), 'filters_respected': int(n_filter_ok)}
print(EXACT)
assert n_exact == 50 and n_filter_ok == 50, 'search results differ from the brute-force answer or break a filter'

sample = rng.choice(N, size=min(500, N), replace=False)
ranks = []
for r in sample:
    hits = index.search(items.title.iloc[r] or items.category_path.iloc[r], k=10)
    pos = np.flatnonzero(hits.item_idx.to_numpy() == items.item_idx.iloc[r])
    ranks.append(int(pos[0]) + 1 if len(pos) else None)
SELF = {'items': int(len(sample)), **{f'recall_at_{k}': round(float(np.mean([x is not None and x <= k for x in ranks])), 3)
                                      for k in (1, 5, 10)}}
print('title self-retrieval:', SELF)
assert SELF['recall_at_10'] >= 0.5, 'titles do not find their own items: the index or the query encoder is wrong'
print('PASS exact search, filters respected, titles find their items')

{'queries': 50, 'same_as_brute_force': 50, 'filters_respected': 50}
title self-retrieval: {'items': 500, 'recall_at_1': 0.856, 'recall_at_5': 0.972, 'recall_at_10': 0.984}
PASS exact search, filters respected, titles find their items


## 8. Example queries (read these by hand)
Typical prototype requests, plus the behaviour the agents rely on: an impossible filter gives an empty result
("no suitable match"), and a misspelled category is rejected with suggestions instead of silently matching nothing.

In [10]:
pd.set_option('display.max_colwidth', 70); pd.set_option('display.width', 200)
EXAMPLES = [
    ('gentle fragrance-free moisturizer for dry sensitive skin', dict(price_max=25)),
    ('vitamin c serum for dark spots', dict(categories=[top_l2[0]] if top_l2 else None)),
    ('mineral sunscreen spf 50', dict(price_max=30)),
    ('lip balm', dict(price_max=10)),
    ('exfoliating body scrub', dict(price_min=10, price_max=40)),
]
example_out = []
for q, f in EXAMPLES:
    f = {k: v for k, v in f.items() if v is not None}
    hits = index.search(q, k=5, **f)
    example_out.append({'query': q, 'filters': f, 'top5': hits[['item_idx', 'title', 'brand', 'price_num', 'score']].to_dict('records')})
    print(f'\n### {q}   {f}')
    print(hits[['rank', 'item_idx', 'score', 'title', 'brand', 'category_path', 'price_num']].to_string(index=False))

empty = index.search('moisturizer', k=5, price_min=10_000)
print('\nImpossible filter (price over $10,000):', len(empty), 'results -> "no suitable match"')
assert len(empty) == 0
try:
    index.search('moisturizer', categories=['Facee'])
    raise AssertionError('a misspelled category must be rejected')
except ValueError as e:
    print('Misspelled category rejected:', e)
print('\nValid level-2 categories for the agent:\n', index.categories(level=2).head(10).to_string(index=False))

t0 = time.time()
for q, f in EXAMPLES * 4:
    index.search(q, k=10, **{k: v for k, v in f.items() if v is not None})
LATENCY_MS = round(1000 * (time.time() - t0) / (len(EXAMPLES) * 4), 1)
t0 = time.time()
for _ in range(50):
    index.search_vector(emb[0], k=10, price_max=25)
VECTOR_MS = round(1000 * (time.time() - t0) / 50, 2)
print(f'\nCPU time per search: {LATENCY_MS} ms with text encoding, {VECTOR_MS} ms for a vector query')


### gentle fragrance-free moisturizer for dry sensitive skin   {'price_max': 25}
 rank  item_idx    score                                                                                                                                                                                                    title                                           brand                                                category_path  price_num
    1     12958 0.738153                                                                 Amazon Basics Ultra Moisturizing Skin Cream for Dry & Sensitive Skin, Dermatologist Tested, Fragrance Free, 16 Ounce (Previously Solimo)                                   Amazon Basics                     Skin Care > Body > Moisturizers > Creams       8.12
    2     24516 0.713483    Tree To Tub Sensitive Skin Face Wash for Dry Skin - Fragrance Free Gentle Face Cleanser for Women & Men, Unscented Hydrating Foaming Facial Cleanser, Daily Face Soap w/All Natural Organic Aloe Vera 

## 9. Optional demo: retrieve products for a user profile
The MiniLM profile vectors from the profile pilot are in the same space as the items, so a profile can be used as
the query directly. The user's own training items are excluded (the prototype never recommends seen items).

In [11]:
PROFILE_DEMO = None
prof_path = PROJECT_ROOT / PILOT_PROFILES
seq_path = DATA_DIR / 'sequences.parquet'
if prof_path.exists() and seq_path.exists():
    assert sha256_file(seq_path) == SEQUENCES_SHA256, 'sequences.parquet is not the frozen version'
    with np.load(prof_path, allow_pickle=False) as z:
        p_uid, p_emb = z['user_idx'], z['embeddings']
    assert p_emb.shape[1] == emb.shape[1], 'profile vectors are not MiniLM vectors'
    seq = pd.read_parquet(seq_path, columns=['user_idx', 'train_items']).set_index('user_idx')
    texts_path = prof_path.with_name('user_profiles.parquet')
    texts = pd.read_parquet(texts_path).set_index('user_idx').profile_text if texts_path.exists() else None
    PROFILE_DEMO = []
    for j in rng.choice(len(p_uid), size=min(3, len(p_uid)), replace=False):
        u = int(p_uid[j]); train = [int(i) for i in seq.train_items[u]]
        hits = index.search_vector(p_emb[j], k=5, exclude_items=train)
        assert not set(hits.item_idx) & set(train)
        PROFILE_DEMO.append({'user_idx': u, 'n_train_items': len(train), 'top5': hits.item_idx.tolist()})
        print(f'\n### user {u} ({len(train)} training items)')
        if texts is not None:
            print('profile:', texts[u][:300])
        print(hits[['rank', 'item_idx', 'score', 'title', 'brand', 'price_num']].to_string(index=False))
    print('\nPASS profile vectors work as queries; training items are excluded')
else:
    print('Pilot profiles or sequences not found; demo skipped.')


### user 17469 (13 training items)
profile: This shopper favors natural and organic brands like Radha Beauty, OZNaturals, and TruSkin, particularly for Vitamin C serums, hyaluronic acid treatments, and cleansers. They also enjoy Olay products, specifically night creams, eye complexes, and cleansing systems, while showing neutral interest in s
 rank  item_idx    score                                                                                                                                                       title        brand  price_num
    1      9447 0.679962                                                                                              Olay Regenerist Cream Cleanser, Regenerating, 5 Ounce (2 Pack)         Olay      22.11
    2      7899 0.673566 Radha Beauty Vitamin C Facial Cleanser, 4 fl. oz - Clear Pores on Oily, Dry & Sensitive Skin, Anti-Aging Herbal Infusion for 8 Times Antioxidant Protection Radha Beauty       9.95
    3      8491 0.673566 Radha Beauty V

## 10. Save the build report

In [12]:
REPORT = {'index_dir': str(OUT_DIR), 'index_name': INDEX_NAME, 'checksums': index.meta['sha256'],
          'stats': STATS, 'same_space_check': SAME_SPACE, 'exact_and_filters': EXACT, 'title_self_retrieval': SELF,
          'examples': example_out, 'cpu_ms_per_text_query': LATENCY_MS, 'cpu_ms_per_vector_query': VECTOR_MS,
          'profile_demo': PROFILE_DEMO, 'uses_reviews_or_interactions': False,
          'built_utc': datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')}
(OUT_DIR / 'build_report.json').write_text(json.dumps(REPORT, indent=2, default=str))
print('Saved', OUT_DIR / 'build_report.json')
print('\nFor Ruchika:\n  from src.retrieval.item_index import ItemIndex\n'
      f"  index = ItemIndex.load('{OUT_DIR}')\n  index.search('gentle moisturizer', k=10, price_max=25, categories=['Face'])")
if IN_COLAB:
    from google.colab import drive
    drive.flush_and_unmount()

Saved /content/drive/.shortcut-targets-by-id/1K2gvVTjU-77GWGMI14ixFt5F57IaijZB/298A/retrieval_index/item_index_minilm_v1/build_report.json

For Ruchika:
  from src.retrieval.item_index import ItemIndex
  index = ItemIndex.load('/content/drive/.shortcut-targets-by-id/1K2gvVTjU-77GWGMI14ixFt5F57IaijZB/298A/retrieval_index/item_index_minilm_v1')
  index.search('gentle moisturizer', k=10, price_max=25, categories=['Face'])
